In [1]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, insert, func
from sqlalchemy.orm import Session
import polars as pl

# Importing from 'app' module
from app.config import db_engine
from app.models import SilverCleanAd, GoldPriceVariationAlert

In [2]:
with db_engine.connect() as connection:
    df_silver_raw = pl.read_database(
        select(
            SilverCleanAd.ad_id,
            SilverCleanAd.title,
            SilverCleanAd.url,
            SilverCleanAd.price,
            SilverCleanAd.date
        ),
        connection=connection
    )

In [3]:
df_gold_market_trends = (
    df_silver_raw
    .sort(['ad_id', 'date'])
    .group_by('ad_id')
    .agg(
        pl.col('title').last().alias('title'),
        pl.col('url').last().alias('url'),
        pl.col('price').first().alias('initial_price'),
        pl.col('price').last().alias('current_price'),
        pl.col('date').first().alias('first_seen_date'),
        pl.col('date').last().alias('last_seen_date')
    )
    # Filter only the ones who had changes
    .filter(pl.col('initial_price') != pl.col('current_price'))
    # Calculates business metrics
    .with_columns(
        (
            pl.col('last_seen_date') - pl.col('first_seen_date')
        ).dt.total_days().cast(pl.Int32).alias('days_on_market'),
        (
            ((pl.col('current_price') - pl.col('initial_price')) / pl.col('initial_price')) * 100
        ).round(2).alias('price_change_pct')
    )
    .drop(['first_seen_date', 'last_seen_date'])
)

In [ ]:
if not df_gold_market_trends.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(GoldPriceVariationAlert), df_gold_market_trends.to_dicts()
        )
else:
    print("No data found to insert.")